In [1]:
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import pandas as pd
import math
import seaborn as sns
import gc
import psutil
import os, random
import posixpath

from pyhdas.frequency import spectrogram, add_db, energy
from pyhdas.aggregate import quantile
from pyhdas.aragon import concat_raw_data, aragon_select_files

from datetime import timedelta, datetime

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
def plot_sound_level(events):
    # loop through event table 
    for _, event in events.iterrows():
        # load the event data
        start, end, poi, label = event["start"], event["end"], event["poi"], event["label_anon"]
        # select the locations 50m here, 50m there 
        poi = np.arange(4150, 4350, 10)
        # if the duration of the event is more than 2 minutes, print the event_label, date and start and end time, and move on to the next row
        event_duration = (end - start).total_seconds()
        if event_duration > 120 or event_duration <= 1:
            continue
    
        # load the strain data based on the start and the end of the event
        start = pd.Timestamp(start)
        end = pd.Timestamp(end)
        start = start.tz_localize("UTC")
        end = end.tz_localize("UTC")
        day = start.day

        dir_data = Path(fr"data/{day}")
        file_list = list(aragon_select_files(dir_data, start, end, extension="bin"))
        if not file_list:
            print(f"The list is empty: {start}, {end}, {label}")
            continue

        ds_raw = concat_raw_data(file_list)
        start = start.tz_localize(None)
        end = end.tz_localize(None)

        # sound level 
        ds_raw = ds_raw.sel(time=slice(start, end), position=poi)
        ds_spect = spectrogram(ds_raw, variable='strain') 
        ds_soundlevel = ds_spect[["Pxx"]].sum(dim="freq")
        ds_soundlevel = add_db(ds_soundlevel)   

        # Plot soundlevel over time and all positions
        fig,ax = plt.subplots(figsize=(15,6))
        ds_soundlevel.Pxx_dB.plot(x="position", y="time", ax=ax, vmax = 40, cmap="magma")
        ax.set_title(f"soundlevel [dB] over all frequencies")

        # save the plot
        output_folder = Path(f"soundlevel_plots/{label}")
        output_folder.mkdir(parents=True, exist_ok=True)
  
        plot_filename = f"{start.strftime('%Y-%m-%d_%H:%M:%S')}_{end.strftime('%H:%M:%S')}_soundlevel.png"
        plt.savefig(output_folder / plot_filename, bbox_inches='tight')

        plt.close(fig)
        
        print(f"Done with {start}, {end}, {label}")
        del ds_raw, ds_spect, ds_soundlevel
        gc.collect()

In [ ]:
def get_sound_normal():
    """
    Get and save sound levels from the normal/silent day.
    """
    all_data = []

    normal_data_dir = "data/19"

    for file in os.listdir(normal_data_dir):

        filepath = [posixpath.join(normal_data_dir, file)]

        # open file one by one
        ds_raw = concat_raw_data(filepath)
        
        # excluding the end of the pipe
        poi = np.arange(1260, 6950, 10)

        ds_raw = ds_raw.sel(position=poi)
        ds_spect = spectrogram(ds_raw, variable='strain')
        ds_soundlevel = ds_spect[["Pxx"]].sum(dim="freq")
        ds_soundlevel = add_db(ds_soundlevel)  

        X_train = ds_soundlevel.Pxx_dB.values  

        if X_train.shape[1] != 59:
            print(f"{file} has dimensions {X_train.shape}")
            continue
        
        all_data.append(X_train)
        print(f"Done with {file}")
    
    all_data_array = np.array(all_data)
    np.save('sound_levels_data.npy', all_data_array)
        

In [ ]:
def get_sound_event_day(day):
    """
    Get and save sound levels from the event day along with the filenames.
    """
    all_data = []

    event_dir = f"data/{day}"

    for file in os.listdir(event_dir):

        if not file.endswith(".bin"):
            continue
        
        try:
            # Parse timestamp from filename
            file_timestamp_str = "_".join(file.split('_')[:4]) 
        except ValueError:
            print(f"Skipping file with bad format: {file}")
            continue

        filepath = [posixpath.join(event_dir, file)]

        # open file one by one
        ds_raw = concat_raw_data(filepath)
        
        # excluding the end of the pipe
        poi = np.arange(1260, 6950, 10)

        ds_raw = ds_raw.sel(position=poi)
        ds_spect = spectrogram(ds_raw, variable='strain')
        ds_soundlevel = ds_spect[["Pxx"]].sum(dim="freq")
        ds_soundlevel = add_db(ds_soundlevel)  

        X_train = ds_soundlevel.Pxx_dB.values  

        if X_train.shape[1] != 59:
            print(f"{file} has shape {X_train.shape}")
            continue
    
        all_data.append((file_timestamp_str, X_train)) 
        print(f"Done with {file}")
    
    dtype = [('filename', 'U256'), ('data', 'O')]
    structured_array = np.array(all_data, dtype=dtype)
    
    np.save(f'{day}_event_day.npy', structured_array)

In [ ]:
# load the event table 
events = pd.read_csv("events_table.csv", parse_dates=["start", "end"])

In [ ]:
# take start time, keep adding one minute until the end time is higher than the end time of the event 
all_data = []

for row, event in events.iterrows(): 

    start, end, poi, label = event["start"], event["end"], event["poi"], event["label_anon"]

    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    start_day = start.day

    end = end.tz_localize("UTC")
    start = start.tz_localize("UTC")

    end_file = start
    dir_data = Path(fr"data/{end.day}")


    while end_file < end:

        end_file = start + pd.Timedelta(seconds=60)

        print(start, end_file)

        # load the file based on the time provided
        file_list = list(aragon_select_files(dir_data, start, end_file, extension="bin"))
        if not file_list:
            print(f"The list is empty: {start}, {end}, {label}")
            continue

        # extract the sound levels 
        ds_raw = concat_raw_data(file_list)

        start = start.tz_localize(None)
        end_file = end_file.tz_localize(None)

        pipe_locs = np.arange(1260, 6950, 10)

        current_length = len(ds_raw.sel(time=slice(start, end_file), position=pipe_locs).time)

        # in case tehre are not enough data points, pad 
        if current_length < 120000:
            milliseconds_to_pad = int(math.ceil((120000 - current_length)/2))
            ds_raw = ds_raw.sel(time=slice(start, end_file+pd.Timedelta(milliseconds=milliseconds_to_pad)), position=pipe_locs)
        else:
            ds_raw = ds_raw.sel(time=slice(start, end_file), position=pipe_locs)
        

        ds_spect = spectrogram(ds_raw, variable='strain') 
        ds_soundlevel = ds_spect[["Pxx"]].sum(dim="freq")
        ds_soundlevel = add_db(ds_soundlevel) 
        
        X = ds_soundlevel.Pxx_dB.values  
        end_file = end_file.tz_localize("UTC")

        start = end_file

        if X.shape[1] != 59:
            print(f"X has shape {X.shape}")
            continue
         # store in the data frame with labels and locations 
    
        all_data.append((row, X, poi, label)) 

    print(f"Done with row {row}")

In [ ]:
dtype = [('row', 'U256'), ('data', 'O'), ('poi', 'U256'), ('label', 'U256')]
structured_array = np.array(all_data, dtype=dtype)
np.save('complete_intrusions.npy', structured_array)